# STEP 3. 데이터 검증
수집된 기상 데이터 품질 확인
- 결측값 / 온도 논리 / 이상값 / 날짜 연속성 / 통계 요약

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

SAVE_DIR  = '/content/drive/MyDrive/JADX_병해충/data'
SAVE_FILE = f'{SAVE_DIR}/tb_weather_pest.csv'

df = pd.read_csv(SAVE_FILE)
df['date']  = pd.to_datetime(df['crtr_ymd'], format='%Y%m%d')
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month

print(f'✅ 로드 완료: {len(df)}건')
print(df.head())

In [ ]:
# 검증 1 — 결측값
print('='*50)
print('  [검증 1] 결측값')
print('='*50)
cols = ['day_avg_tp','day_hghst_tp','day_lowst_tp']
for stn, grp in df.groupby('stn_nm'):
    nulls = grp[cols].isnull().sum()
    status = '✅' if nulls.sum()==0 else '⚠️'
    print(f'  {status} {stn}')
    for col, cnt in nulls.items():
        if cnt > 0:
            dates = grp[grp[col].isnull()]['crtr_ymd'].tolist()
            print(f'     - {col}: {cnt}건 → {dates}')

In [ ]:
# 검증 2 — 온도 논리
print('='*50)
print('  [검증 2] 온도 논리 (최고 > 평균 > 최저)')
print('='*50)
invalid = df[(df['day_hghst_tp'] < df['day_avg_tp']) | (df['day_avg_tp'] < df['day_lowst_tp'])]
if len(invalid)==0:
    print('  ✅ 이상 없음')
else:
    print(f'  ⚠️ 위반: {len(invalid)}건')
    print(invalid[['crtr_ymd','stn_nm','day_avg_tp','day_hghst_tp','day_lowst_tp']])

In [ ]:
# 검증 3 — 이상값
print('='*50)
print('  [검증 3] 이상값 (최고>40℃ / 최저<-5℃)')
print('='*50)
abnormal = df[(df['day_hghst_tp'] > 40) | (df['day_lowst_tp'] < -5)]
if len(abnormal)==0:
    print('  ✅ 이상값 없음')
else:
    print(f'  ⚠️ {len(abnormal)}건')
    print(abnormal[['crtr_ymd','stn_nm','day_avg_tp','day_hghst_tp','day_lowst_tp']])

In [ ]:
# 검증 4 — 날짜 연속성
print('='*50)
print('  [검증 4] 날짜 연속성')
print('='*50)
for stn, grp in df.groupby('stn_nm'):
    dates = pd.to_datetime(grp['crtr_ymd'], format='%Y%m%d').sort_values()
    missing = pd.date_range(dates.min(), dates.max()).difference(dates)
    status = '✅' if len(missing)==0 else '⚠️'
    print(f'  {status} {stn}: 누락 {len(missing)}일')
    if len(missing) > 0:
        print(f'     {[d.strftime("%Y%m%d") for d in missing]}')

In [ ]:
# 검증 5 — 월평균 기온 그래프
monthly = df.groupby(['stn_nm','month'])['day_avg_tp'].mean().reset_index()
fig, ax = plt.subplots(figsize=(12,5))
for stn, grp in monthly.groupby('stn_nm'):
    ax.plot(grp['month'], grp['day_avg_tp'], marker='o', label=stn)
ax.axhline(y=9.8, color='red', linestyle='--', alpha=0.5, label='발육 하한온도 9.8℃')
ax.set_xlabel('월')
ax.set_ylabel('평균기온 (℃)')
ax.set_title('제주 관측소별 월평균 기온 (2020~2025)')
ax.set_xticks(range(1,13))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/validation_monthly_temp.png', dpi=150)
plt.show()
print('✅ 그래프 저장 완료')

In [ ]:
# 최종 요약
print('='*50)
print('  검증 최종 요약')
print('='*50)
checks = {'결측값': df[cols].isnull().sum().sum(), '온도 논리 위반': len(invalid), '이상값': len(abnormal)}
total = 0
for k, v in checks.items():
    print(f'  {"✅" if v==0 else "⚠️"} {k}: {v}건')
    total += v
print(f'\n  총 이슈: {total}건')
print('  ✅ STEP 4 진행 가능' if total==0 else '  ⚠️ 이슈 확인 후 진행')